# Exploratory Data Analysis: Loan Approval Prediction

This notebook analyzes the Kaggle `architsharma01/loan-approval-prediction-dataset`.
We investigate data hygiene, strip whitespace artifacts, check class balance, and study feature relationships and potential leakage.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path("../data/raw/loan_approval_dataset.csv")
df_raw = pd.read_csv(data_path)
print("Raw shape:", df_raw.shape)
print("Raw columns with leading whitespace:", repr(list(df_raw.columns)))

Raw shape: (4269, 13)
Raw columns with leading whitespace: ['loan_id', ' no_of_dependents', ' education', ' self_employed', ' income_annum', ' loan_amount', ' loan_term', ' cibil_score', ' residential_assets_value', ' commercial_assets_value', ' luxury_assets_value', ' bank_asset_value', ' loan_status']


## 1. Data Cleaning & Normalization
- Column names and text fields contain leading spaces.
- Drop `loan_id` as it is an arbitrary sequence number.

In [2]:
df = df_raw.copy()
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()
df = df.drop(columns=['loan_id'], errors='ignore')
print("Clean columns:", list(df.columns))
print("Missing values:", df.isnull().sum().to_dict())
print("Duplicate rows:", df.duplicated().sum())

Clean columns: ['no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']
Missing values: {'no_of_dependents': 0, 'education': 0, 'self_employed': 0, 'income_annum': 0, 'loan_amount': 0, 'loan_term': 0, 'cibil_score': 0, 'residential_assets_value': 0, 'commercial_assets_value': 0, 'luxury_assets_value': 0, 'bank_asset_value': 0, 'loan_status': 0}
Duplicate rows: 0


## 2. Class Balance Analysis

In [3]:
print("Class counts:")
print(df['loan_status'].value_counts())
print("\nClass proportions:")
print(df['loan_status'].value_counts(normalize=True))

Class counts:
loan_status
Approved    2656
Rejected    1613
Name: count, dtype: int64

Class proportions:
loan_status
Approved    0.62216
Rejected    0.37784
Name: proportion, dtype: float64


## 3. CIBIL Score and Target Relationship (Feature Dominance / Leakage Analysis)

In synthetic financial datasets, credit score is often generated with near-deterministic rules. Let us measure this separation.

In [4]:
print("cibil_score summary by loan_status:")
print(df.groupby('loan_status')['cibil_score'].describe())
df['target_numeric'] = (df['loan_status'] == 'Approved').astype(int)
corr = df[['cibil_score', 'target_numeric']].corr().iloc[0, 1]
print(f"\nCorrelation between CIBIL score and approval: {corr:.4f}")
threshold_acc = ((df['cibil_score'] >= 550) == df['target_numeric']).mean()
print(f"Approval accuracy using threshold (CIBIL >= 550) alone: {threshold_acc:.4%}")

cibil_score summary by loan_status:
              count        mean         std    min    25%    50%    75%    max
loan_status                                                                   
Approved     2656.0  703.461973  125.249016  300.0  618.0  711.0  803.0  900.0
Rejected     1613.0  429.468072   78.401752  300.0  364.0  429.0  493.0  885.0

Correlation between CIBIL score and approval: 0.7705
Approval accuracy using threshold (CIBIL >= 550) alone: 95.3619%


### Conclusion on Model Accuracy
The CIBIL score feature alone explains over 95% of target variance. This explains why tree-based models and logistic regression can easily achieve 97-98%+ accuracy on this Kaggle dataset. We acknowledge this transparently as synthetic target dependence rather than unexpected data leakage in production.